## Introduction
lorem ipsum

## 0.0 - 
Imports

In [28]:
import pandas as pd

## 0.0 - 
Load the relevant datasets

In [29]:
# Load merged_df from preprocessing:
merged_df = pd.read_csv('merged_df.csv')

# Load in the third dataset for inspection
dataset3 = pd.read_csv('../raw_data/Consumer_meta.csv', skiprows = 1)

# Create copies of the original daatsets preseving the originals
df1 = dataset3.copy()
df2 = merged_df.copy()

## 0.0 -
Following the same steps as dataset one and two, preprocesing is going to be the same for dataset 3

### 0.1 - Dataset shape

In [30]:
# Print the shape of dataset 3:
print("Dataset Three Shape:", df1.shape)

Dataset Three Shape: (20000, 20)


20,000 by 20 observations gives us a much more limited dataset to work with as opposed to the previous datasets. This wont be an issue considering we will be deploying SDV on the data, so it is only ratios we need to figure out. The key issue is being much more careful in the preprocessing steps. Like removing null values - as removing too mant null values can effect the integrity of the dataset. 

### 0.2 - Health Summary
why are you using a health summary

In [31]:
# Initlise the function: 
def health_summary(df):
    report = pd.DataFrame({
        'Data Type  ': df.dtypes,
        'Null Values  ': df.isnull().sum(),
        'Unique Values  ': df.nunique(),
        'Duplicate Values   ': df.duplicated().sum()
    })
    return report

# Print the health summaries for both dataframes:
print("-----------     Dataset Three Health Summary     ------------")
display(health_summary(df1))

-----------     Dataset Three Health Summary     ------------


,Data Type,Null Values,Unique Values,Duplicate Values
transaction_id,int64,0,20000,0
timestamp,str,0,19992,0
store_id,int64,0,45,0
city,str,0,9,0
country,str,0,4,0
store_type,str,0,3,0
product_category,str,0,6,0
product_name,str,0,43,0
unit_price,float64,0,359,0
quantity,int64,0,9,0


brief desc of what was found within the health summary:

## 2.0 - Identifying Valuable Features
Explain paragraph - Essentially there is no point preprocessing features that are going to be dropped/ not included. 

Explain what you are looking for in the ideal feature and why (What is the strategy)

The key here isnt about determining what features here are valuable, but more a question about what merged_df lacks and could benifit from. 

### 2.1 - Print the list of features

In [32]:
# Implement a function that prints features and values:
def table_inspection(df):
    for col in df.columns:
        print("Column Title:", col, " - ","Example Value:", df[col].iloc[0])
# Call the above function:
print(table_inspection(df1))

Column Title: transaction_id  -  Example Value: 10001
Column Title: timestamp  -  Example Value: 2023-01-01 00:39:39
Column Title: store_id  -  Example Value: 35
Column Title: city  -  Example Value: Melbourne
Column Title: country  -  Example Value: AUS
Column Title: store_type  -  Example Value: Mall Kiosk
Column Title: product_category  -  Example Value: Coffee
Column Title: product_name  -  Example Value: Double Espresso
Column Title: unit_price  -  Example Value: 3.04
Column Title: quantity  -  Example Value: 1
Column Title: discount_applied  -  Example Value: True
Column Title: payment_method  -  Example Value: Credit Card
Column Title: customer_id  -  Example Value: qiyfrwsk
Column Title: customer_age_group  -  Example Value: 35-44
Column Title: customer_gender  -  Example Value: Female
Column Title: loyalty_member  -  Example Value: False
Column Title: weather_condition  -  Example Value: Sunny
Column Title: temperature_c  -  Example Value: 22.2
Column Title: holiday_name  -  E

## 2.3 - Table Inspection Analysis
From the above table we can start to 'whittle down' features that do not provide any relevance to consumer segementation: 

1. 'transaction_id' - merged_df already contains transaction id's through its 'Invoice' and 'customer_id' features. This can be dropped. 

2. 'timestamp' - merged_df already contains timestamp featues that have already been parsed and split into their own numerical features, this can also be dropped. 

3. 'store_id' - merged_df does not currently contain any data like this. This feature will be kept for now but further inspection will be completed to assess the relevance of the data.

4. 'city' - Again merged_df does not contain data like this. However only one feature describing location will be needed, so this decision will be out of this feature and 'store_id' not both. 

5. 'country' - Similar to merged_df, having data that describes international locations is irrelevant for the context of this project and only serves to expand scope unnessicerilly. This feature will be dropped. 

6. 'store_type' - 

7. 'product_category' - This feature will be very useful in helping enrich consumer profiling as specific and unique products can be singled out and provide a comprehensive purchasing record. 

8. 'product_name' - This category will go hand-in-hand with 'product_category' and will also help to enrich data and produce deeper more meaningful segementation. However inspection will need to be carried oiut in order to assess how these features can be reasonably converted into numerical datatypes. 

9. 'unit_price' - This feature introduces some complication. merged_df contains a 'Price' feature already however if we are using product category and product name from this specific dataset, it might be wise to keep price as all; three of these features will be related. This feature will be kept for now but will require further inspection. 

10. 'quantity' - Again an assessment into how these products are bought might be better from this dataset than the original already found in merged_df. 

11. 'discount_applied' - keep

12. 'payment_method' - keep 

13. 'customer_id' - remove

14. 'customer_age_group' - keep

15. 'customer_gender' - keep

16. 'loyalty_member' - keep

17. 'weather_condtion' - remove

18. 'temperature_c' - remove

19. 'holiday_name' - remove

20. 'total_amount' - keep

### 2.4 - Removing Unneeded Features:

In [33]:
drop_features = ['transaction_id',
                'timestamp',
                'country', 
                'customer_id',
                'weather_condition',
                'temperature_c',
                'holiday_name']

df1 = df1.drop(columns=drop_features)

# Confirm feature removal was successful:
print(table_inspection(df1))

Column Title: store_id  -  Example Value: 35
Column Title: city  -  Example Value: Melbourne
Column Title: store_type  -  Example Value: Mall Kiosk
Column Title: product_category  -  Example Value: Coffee
Column Title: product_name  -  Example Value: Double Espresso
Column Title: unit_price  -  Example Value: 3.04
Column Title: quantity  -  Example Value: 1
Column Title: discount_applied  -  Example Value: True
Column Title: payment_method  -  Example Value: Credit Card
Column Title: customer_age_group  -  Example Value: 35-44
Column Title: customer_gender  -  Example Value: Female
Column Title: loyalty_member  -  Example Value: False
Column Title: total_amount  -  Example Value: 2.74
None


### 2.5 - Further feature analysis:
As previously stated, more analysis needs to be carried out on specific fetures in order to responsibly determine if they should be dropped. 

### 2.6 - Store Location
It would be helpfil for our data to contain certain location data within the dataset. This data does not need to be expansive - only a variation of three locations is needed. Below code block inspects store_id, city and store_type in order to determine the most suitable choice - if any. 

In [37]:
store_inspection = ['store_id',
                    'store_type',
                    'city']

def value_inspection(df):
    for col in store_inspection:
        print("Feature:", col, " - ", df[col].unique())
        print("-----------------------------------------------")

print(value_inspection(df1))

Feature: store_id  -  [35 25 23 38  9 42 12 21 27  7 40  1 20 28  5  6 30 43  4 18 17 11 19 24
 34 15 31 26  8  3 33 45 22 37 41 39 36 10 14 44  2 32 13 16 29]
-----------------------------------------------
Feature: store_type  -  <StringArray>
['Mall Kiosk', 'Standalone', 'Airport']
Length: 3, dtype: str
-----------------------------------------------
Feature: city  -  <StringArray>
[  'Melbourne',  'Manchester',     'Toronto', 'Los Angeles',   'Vancouver',
     'Chicago',      'Sydney',    'New York',      'London']
Length: 9, dtype: str
-----------------------------------------------
None


As we can see, the feature that matches the context of ou r coffee shop the best is store _type. store_id doesnt actually provide ant tangiable informaiton, and the behaviour seen within 'city' is out of scope for a coffee shop with 3 locations in one city. Therefore store_type is the most reasonable. 

In [39]:
df1 = df1.drop(columns=['city', 'store_id'])